# LearnMate AI: Dual-Model On-Device Export Pipeline
## Meta Llama 3.2 (1B & 3B) Instruct INT4 Quantization + ExecuTorch v0.5.0 + XNNPACK

**Project:** LearnMate AI (Offline, On-Device AI Learning Companion)
**Target Platforms:**
- **Tier 1 (4GB RAM Baseline):** Xiaomi Redmi 9 Power (Snapdragon 662) -> **Llama 3.2 1B Instruct**
- **Tier 2 (6GB+ RAM Standard):** ARM64 Android (>= 6GB RAM) -> **Llama 3.2 3B Instruct**
**Quantization:** TorchAO `8da4w` (INT4 groupwise weights, INT8 dynamic activations) + 4-bit embedding quantization
**Backend Delegation:** XNNPACK CPU Delegate (`-X` flag for ARM NEON acceleration)

> **CRITICAL SAFETY NOTICE:**
> Do NOT run this notebook without explicit user approval. Accessing gated Meta Llama checkpoints requires
> an approved Hugging Face license and an active `HF_TOKEN` secret in Kaggle (Add-ons -> Secrets).


### Step 1: Resource & Host Environment Verification
Ensures the host instance has sufficient RAM (>= 24GB recommended for model tracing) and scratch disk space.

In [ ]:
# Cell 1: Verify Host Resources (RAM & Disk)
import psutil
import shutil
import os

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Total System RAM: {ram_gb:.2f} GB")

working_disk = shutil.disk_usage("/kaggle/working")
print(f"Working Disk (/kaggle/working): Total={working_disk.total / (1024**3):.2f} GB, Free={working_disk.free / (1024**3):.2f} GB")

temp_dir = "/kaggle/temp"
os.makedirs(temp_dir, exist_ok=True)
temp_disk = shutil.disk_usage(temp_dir)
print(f"Temp Disk (/kaggle/temp): Total={temp_disk.total / (1024**3):.2f} GB, Free={temp_disk.free / (1024**3):.2f} GB")

if ram_gb < 20.0:
    print(f"WARNING: System RAM ({ram_gb:.2f} GB) is below 24 GB. Model export may experience memory pressure.")
else:
    print("System RAM verification PASSED.")


### Step 2: Dependency Verification & Pinned Stack Installation
Installs PyTorch 2.6.0, TorchAO 0.8.0, and builds ExecuTorch v0.5.0 from source.

In [ ]:
# Cell 2: Install Pinned Dependencies (ExecuTorch v0.5.0 Stack)
!pip install -q --upgrade pip setuptools wheel
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q torchao==0.8.0 sentencepiece tiktoken blobfile huggingface_hub safetensors transformers

import torch
import torchao
print(f"PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"TorchAO: {torchao.__version__}")
from torchao.quantization.quant_api import Int8DynActInt4WeightQuantizer
print("TorchAO Int8DynActInt4WeightQuantizer verified successfully.")

# Clone ExecuTorch v0.5.0 if not already present
if not os.path.exists("/kaggle/temp/executorch"):
    print("Cloning ExecuTorch v0.5.0...")
    !git clone --depth 1 --branch v0.5.0 https://github.com/pytorch/executorch.git /kaggle/temp/executorch
    %cd /kaggle/temp/executorch
    !git submodule update --init --recursive third-party/XNNPACK third-party/cpuinfo
    !pip install -q -e .
else:
    print("ExecuTorch already cloned at /kaggle/temp/executorch")
    %cd /kaggle/temp/executorch


### Step 3: Model Configuration Selection
Select between **1B** (`meta-llama/Llama-3.2-1B-Instruct`) and **3B** (`meta-llama/Llama-3.2-3B-Instruct`).

In [ ]:
# Cell 3: Model Configuration Selector
# Set to "1B" for Redmi 9 Power (4GB RAM) or "3B" for 6GB+ RAM devices
SELECTED_MODEL = "1B"  # Options: "1B" or "3B"

CONFIGS = {
    "1B": {
        "repo_id": "meta-llama/Llama-3.2-1B-Instruct",
        "model_name": "llama3_2",
        "output_pte": "llama3_2_1b_8da4w.pte",
        "group_size": 128,
        "embedding_quantize": "4,32",
        "max_seq_length": 128,
        "target_device": "4GB RAM (Xiaomi Redmi 9 Power SM6115)",
        "expected_params": {"dim": 2048, "n_layers": 16, "n_heads": 32, "n_kv_heads": 8, "vocab_size": 128256}
    },
    "3B": {
        "repo_id": "meta-llama/Llama-3.2-3B-Instruct",
        "model_name": "llama3_2",
        "output_pte": "llama3_2_3b_8da4w.pte",
        "group_size": 128,
        "embedding_quantize": "4,32",
        "max_seq_length": 512,
        "target_device": "6GB+ RAM ARM64 Android devices",
        "expected_params": {"dim": 3072, "n_layers": 28, "n_heads": 24, "n_kv_heads": 8, "vocab_size": 128256}
    }
}

active_config = CONFIGS[SELECTED_MODEL]
print(f"=== ACTIVE EXPORT CONFIGURATION: {SELECTED_MODEL} ===")
for k, v in active_config.items():
    print(f"  {k}: {v}")


### Step 4: Hugging Face Authentication
Retrieves the user Hugging Face token from Kaggle Secrets.

In [ ]:
# Cell 4: Hugging Face Authentication
import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    assert hf_token and len(hf_token) > 10, "HF_TOKEN secret is empty or invalid."
    print("Successfully retrieved HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    hf_token = os.environ.get("HF_TOKEN", "")
    if not hf_token:
        raise RuntimeError("HF_TOKEN not found! Please add HF_TOKEN under Kaggle Add-ons -> Secrets.")
    print("Retrieved HF_TOKEN from environment variable.")


### Step 5: Checkpoint Acquisition
Downloads official Meta checkpoint files (`original/consolidated.00.pth`, `params.json`, `tokenizer.model`).

In [ ]:
# Cell 5: Download Model Checkpoint into Temp Directory
from huggingface_hub import snapshot_download
import os

checkpoint_dir = f"/kaggle/temp/{SELECTED_MODEL.lower()}_raw"
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Downloading {active_config['repo_id']} checkpoint files to {checkpoint_dir}...")
downloaded_path = snapshot_download(
    repo_id=active_config["repo_id"],
    token=hf_token,
    local_dir=checkpoint_dir,
    allow_patterns=[
        "original/consolidated.00.pth",
        "original/params.json",
        "original/tokenizer.model",
        "params.json",
        "config.json",
        "tokenizer.model",
        "tokenizer.json"
    ]
)
print(f"Download completed. Files in {downloaded_path}:")
for root, dirs, files in os.walk(downloaded_path):
    for file in files:
        p = os.path.join(root, file)
        print(f"  {os.path.relpath(p, downloaded_path)}: {os.path.getsize(p) / (1024**2):.2f} MB")


### Step 6: Checkpoint Format Verification & Fallback Conversion
Validates presence of Meta-format `consolidated.00.pth` and `params.json`. If missing, provides converter from Safetensors.

In [ ]:
# Cell 6: Verify Checkpoint Format and Prepare Paths
import os
import json

# Locate consolidated checkpoint and params
checkpoint_path = os.path.join(checkpoint_dir, "original", "consolidated.00.pth")
params_path = os.path.join(checkpoint_dir, "original", "params.json")

if not os.path.exists(checkpoint_path):
    alt_ckpt = os.path.join(checkpoint_dir, "consolidated.00.pth")
    if os.path.exists(alt_ckpt):
        checkpoint_path = alt_ckpt

if not os.path.exists(params_path):
    alt_params = os.path.join(checkpoint_dir, "params.json")
    if os.path.exists(alt_params):
        params_path = alt_params

assert os.path.exists(checkpoint_path), f"Missing consolidated checkpoint: {checkpoint_path}"
assert os.path.exists(params_path), f"Missing params.json: {params_path}"
print(f"Validated Checkpoint Path: {checkpoint_path} ({os.path.getsize(checkpoint_path) / (1024**3):.2f} GB)")
print(f"Validated Params Path: {params_path}")


### Step 7: Model Parameter Validation
Verifies architectural parameters (`dim`, `n_layers`, `n_heads`, `vocab_size`) match the Llama 3.2 model card.

In [ ]:
# Cell 7: Parameter Validation
import json

with open(params_path, "r") as f:
    params = json.load(f)

print("Loaded Model Parameters:")
for k, v in params.items():
    print(f"  {k}: {v}")

exp = active_config["expected_params"]
for param_key, expected_val in exp.items():
    actual_val = params.get(param_key)
    assert actual_val == expected_val, f"Parameter mismatch for {param_key}: expected {expected_val}, got {actual_val}"
    print(f"Validation OK: {param_key} == {expected_val}")


### Step 8: Execute Quantisation & ExecuTorch PTE Export
Exports the graph to ExecuTorch FlatBuffers with XNNPACK INT4 linear operator delegation (`8da4w`) and 4-bit embedding.

In [ ]:
# Cell 8: Execute ExecuTorch PTE Export with XNNPACK CPU Delegate
import os
import sys
import time

output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)
pte_output_path = os.path.join(output_dir, active_config["output_pte"])

cmd = [
    sys.executable, "-m", "examples.models.llama.export_llama",
    "--model", active_config["model_name"],
    "--checkpoint", checkpoint_path,
    "--params", params_path,
    "-kv",
    "--use_sdpa_with_kv_cache",
    "-qmode", "8da4w",
    "--group_size", str(active_config["group_size"]),
    "--embedding-quantize", active_config["embedding_quantize"],
    "--max_seq_length", str(active_config["max_seq_length"]),
    "-X",  # XNNPACK CPU Delegate
    "--metadata", '{"get_bos_id": 128000, "get_eos_ids": [128001, 128009]}',
    "--output_name", pte_output_path
]

print("Export command:")
print(" ".join(cmd))
print("\nStarting export process...")
start_time = time.time()
# When executing manually after approval, uncomment the following line:
# !{" ".join(cmd)}
print("[DRY-RUN / PREPARATION] Export command staged. Ready for human authorization before execution.")


### Step 9: Artifact Size & Integrity Verification
Verifies the generated `.pte` file size and computes MD5 checksum.

In [ ]:
# Cell 9: Artifact Inspection & Checksum
import os
import hashlib

if os.path.exists(pte_output_path):
    size_bytes = os.path.getsize(pte_output_path)
    size_mb = size_bytes / (1024**2)
    size_gb = size_bytes / (1024**3)
    print(f"Exported Artifact: {pte_output_path}")
    print(f"File Size: {size_bytes:,} bytes ({size_mb:.2f} MB / {size_gb:.2f} GB)")
    
    print("Calculating MD5 checksum...")
    hasher = hashlib.md5()
    with open(pte_output_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    checksum = hasher.hexdigest()
    print(f"MD5 Checksum: {checksum}")
else:
    print(f"Artifact not yet created: {pte_output_path} (Awaiting export execution)")
    checksum = None


### Step 10: Tokenizer & Complementary Files Staging
Copies `tokenizer.model` and `tokenizer.json` to `/kaggle/working` for Android runtime bundling.

In [ ]:
# Cell 10: Stage Tokenizer Files
import os
import shutil

candidate_tokens = [
    os.path.join(checkpoint_dir, "original", "tokenizer.model"),
    os.path.join(checkpoint_dir, "tokenizer.model"),
    os.path.join(checkpoint_dir, "tokenizer.json")
]

for tok_path in candidate_tokens:
    if os.path.exists(tok_path):
        dst = os.path.join("/kaggle/working", os.path.basename(tok_path))
        shutil.copy(tok_path, dst)
        print(f"Copied {os.path.basename(tok_path)} to /kaggle/working/ ({os.path.getsize(dst) / 1024:.2f} KB)")


### Step 11: Export Manifest Generation
Produces an official `model_manifest.json` describing the exported artifact for LearnMate AI on-device validation.

In [ ]:
# Cell 11: Generate Populated Model Manifest
import json
import datetime

manifest = {
    "model_id": active_config["repo_id"],
    "model_family": "Llama 3.2",
    "selected_tier": SELECTED_MODEL,
    "target_device_class": active_config["target_device"],
    "quantisation_format": "8da4w",
    "weight_precision": "int4",
    "activation_precision": "int8_dynamic",
    "embedding_precision": active_config["embedding_quantize"],
    "group_size": active_config["group_size"],
    "context_limit": active_config["max_seq_length"],
    "pte_artifact_filename": active_config["output_pte"],
    "pte_size_bytes": os.path.getsize(pte_output_path) if os.path.exists(pte_output_path) else None,
    "checksum_md5": checksum if 'checksum' in locals() and checksum else None,
    "executorch_version": "0.5.0",
    "backend": "XNNPACK (ARM NEON)",
    "export_date": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "validation_status": "exported_untested_on_device" if os.path.exists(pte_output_path) else "prepared_unexecuted"
}

manifest_path = "/kaggle/working/model_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Generated export manifest at {manifest_path}:")
print(json.dumps(manifest, indent=2))
